### Connect to ElasticSearch

In [34]:
from pprint import pprint
from elasticsearch import Elasticsearch

In [2]:
es = Elasticsearch('http://localhost:9200')
client_info = es.info()
pprint(client_info)
pprint('Connected to Elasticsearch successfully!')
pprint(client_info.body)

ObjectApiResponse({'name': 'c813a54bbd9a', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'WGXTdf8bTw6Y1ejhBBncsA', 'version': {'number': '8.15.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '1a77947f34deddb41af25e6f0ddb8e830159c179', 'build_date': '2024-08-05T10:05:34.233336849Z', 'build_snapshot': False, 'lucene_version': '9.11.1', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})
'Connected to Elasticsearch successfully!'
{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'WGXTdf8bTw6Y1ejhBBncsA',
 'name': 'c813a54bbd9a',
 'tagline': 'You Know, for Search',
 'version': {'build_date': '2024-08-05T10:05:34.233336849Z',
             'build_flavor': 'default',
             'build_hash': '1a77947f34deddb41af25e6f0ddb8e830159c179',
             'build_snapshot': False,
             'build_type': 'docker',
             'lucene_version': '9.11.1',
             'minimum_index_compatib

Shards and replicas are important settings for an Elasticsearch index that determine how data is distributed and replicated across the cluster.
- Shards: An index can be divided into multiple shards, which are essentially smaller pieces of the index. Each shard is a self-contained index that can be stored on a different node in the cluster. The number of shards determines how the data is distributed across the cluster and can affect search performance and scalability. A common default is 5 shards, but this can be adjusted based on the expected size of the index and the workload.
- Replicas: Replicas are copies of the primary shards. They provide redundancy and improve search performance by allowing queries to be served from multiple nodes. The number of replicas determines how many copies of each shard are created. A common default is 1 replica, which means that there will be one copy of each primary shard. This can be adjusted based on the desired level of fault tolerance and read performance.   

### Insert one document

#### Create a dummy index just to insert one document

In [7]:
es.indices.delete(index='my_index', ignore_unavailable=True)
# specify the number of replicas and shards for the index
es.indices.create(index='my_index', settings={"index": {
    'number_of_shards': 3,
     'number_of_replicas': 2}},)


ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_index'})

In [8]:
document = { 'title':'title',
             'text': 'text',
             'created_on':'2026-05-12',
           }
response = es.index(index = 'my_index', body = document)
response        

ObjectApiResponse({'_index': 'my_index', '_id': 'SnV0HJ4Brir_WdooQQxs', '_version': 1, 'result': 'created', '_shards': {'total': 3, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1})

The `response` object contains the result of the operation. If we successfully inserted the document, then `result = created`. Each document has an `id` and is fragmented into `shards`

In [10]:
print(response['result'])

created


In [11]:
print(response['_shards'])

{'total': 3, 'successful': 1, 'failed': 0}


In [13]:
print(response['_id'])

SnV0HJ4Brir_WdooQQxs


In [14]:
print(response['_index'])

my_index


### Insert multiple documents

In [26]:
import json
dummy_data = json.load(open("data/dummy_data.json"))
dummy_data

[{'title': 'Title 1', 'text': 'Description 1', 'created_on': '2026-05-01'},
 {'title': 'Title 2', 'text': 'Description 2', 'created_on': '2026-05-02'},
 {'title': 'Title 3', 'text': 'Description 3', 'created_on': '2026-05-03'},
 {'title': 'Title 4', 'text': 'Description 4', 'created_on': '2026-05-04'},
 {'title': 'Title 5', 'text': 'Description 5', 'created_on': '2026-05-05'}]

In [27]:
def insert_document(document):
    response = es.index(index = 'my_index', body = document)
    return response

def print_info(response):
    print(f"""Document ID: {response['_id']} is '{response['result']}' and is split into {response['_shards']['total']} shards.""")
    
for document in dummy_data:
    response = insert_document(document)
    print_info(response)
    

Document ID: TnWQHJ4Brir_WdooPQxD is 'created' and is split into 3 shards.
Document ID: T3WQHJ4Brir_WdooPQyB is 'created' and is split into 3 shards.
Document ID: UHWQHJ4Brir_WdooPQyW is 'created' and is split into 3 shards.
Document ID: UXWQHJ4Brir_WdooPQzQ is 'created' and is split into 3 shards.
Document ID: UnWQHJ4Brir_WdooPQzg is 'created' and is split into 3 shards.


### Print mapping

In [31]:
from pprint import pprint

index_mapping = es.indices.get_mapping(index = 'my_index')
pprint(index_mapping['my_index']['mappings']['properties'])

{'created_on': {'type': 'date'},
 'text': {'fields': {'keyword': {'ignore_above': 256, 'type': 'keyword'}},
          'type': 'text'},
 'title': {'fields': {'keyword': {'ignore_above': 256, 'type': 'keyword'}},
           'type': 'text'}}


### Manual mapping

In [33]:
es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(index='my_index')

mapping = {
    'properties': {
        'created_on': {'type': 'date'},
        'text': {
            'type': 'text',
            'fields': {
                'keyword': {
                    'type': 'keyword',
                    'ignore_above': 256
                }
            }
        },
        'title': {
            'type': 'text',
            'fields': {
                'keyword': {
                    'type': 'keyword',
                    'ignore_above': 256
                }
            }
        }
    }
}

es.indices.put_mapping(index='my_index', body=mapping)

index_mapping = es.indices.get_mapping(index='my_index')
pprint(index_mapping["my_index"]["mappings"]["properties"])

{'created_on': {'type': 'date'},
 'text': {'fields': {'keyword': {'ignore_above': 256, 'type': 'keyword'}},
          'type': 'text'},
 'title': {'fields': {'keyword': {'ignore_above': 256, 'type': 'keyword'}},
           'type': 'text'}}
